# Paper-1 Runner — ejecución secuencial de las 5 fases

Notebook para ejecutar en vivo las 5 fases del Paper-1 viendo el progreso en stdout.

**Fases en orden obligatorio:**
1. `phase1_train_hmm_seeds.py` — entrena HMM K∈{3..10} × 3 seeds (copia legacy seed=42 + entrena seeds 2021 y 7).
2. `phase2_k_sweep_seeds.py` — K-sweep downstream con 3 seeds. Output: `scripts/paper1/k_optimal.json`.
3. `phase3_plan_a_seeds.py` — Plan A completo 6 técnicas × 4 datasets × 4 horizontes × 3 seeds.
4. `phase4_cross_domain_seeds.py` — Cross-domain Traffic/Exchange × 4 fuentes HMM × 3 seeds.
5. `phase5_intrinsic_metrics.py` — métricas intrínsecas 6 técnicas × 4 datasets × 3 seeds.

**Todas las fases son resumibles** (skip si output existe). Puedes interrumpir y reanudar con la misma celda.

**Plan completo**: ver `memoria/secciones/plan_paper.md`.

---

## Estimación de coste CPU total

| Fase | Experimentos nuevos | Tiempo CPU aprox. |
|---|---|---|
| Phase 1 (HMM train) | 64 Baum-Welch nuevos (seed=42 se copia) | ~6-10 h |
| Phase 2 (K-sweep) | 192 Transformer trains | ~15-25 h |
| Phase 3 (Plan A) | 288 Transformer trains | ~60-80 h |
| Phase 4 (Cross-domain) | 108 Transformer trains | ~25-35 h |
| Phase 5 (Intrinsic) | ~30 min total | ~30 min |
| **Total** | **~652 experimentos** | **~110-150 h CPU** |

En paralelo / con GPU los tiempos bajan 3-5x.

## 0. Setup — verificar repo, caches existentes, entorno

In [ ]:
import os, sys
from pathlib import Path

REPO = '/home/jaime/TFG/RITMO'
if os.getcwd() != REPO:
    os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

print('CWD:', os.getcwd())
print('Python:', sys.executable)

# Verificar que los 4 CSVs existen.
from scripts.paper1.config import DATASETS, CROSS_DOMAIN_TARGETS, SEEDS, K_VALUES
print(f'\nSeeds: {SEEDS}')
print(f'K values: {K_VALUES}')
print(f'\nDatasets IN-DOMAIN:')
for ds in DATASETS:
    p = Path(ds['csv'])
    print(f"  {ds['name']:12} {ds['csv']:50} exists={p.exists()}")
print(f'\nDatasets CROSS-DOMAIN:')
for t in CROSS_DOMAIN_TARGETS:
    p = Path(t['root']) / t['data_path']
    print(f"  {t['name']:12} {str(p):50} exists={p.exists()}")

In [ ]:
# Verificar caches HMM legacy disponibles (seed=42 implícita).
from scripts.paper1.config import hmm_cache_path_legacy, K_VALUES, DATASETS

missing = []
for ds in DATASETS:
    for K in K_VALUES:
        p = Path(hmm_cache_path_legacy(ds['name'], K))
        if not p.exists():
            missing.append((ds['name'], K, str(p)))

if missing:
    print(f'FALTAN {len(missing)} caches legacy (se entrenarán desde cero con seed=42 en phase1):')
    for m in missing[:10]:
        print(f'  {m}')
else:
    print(f'OK: las 32 caches legacy (4 datasets × 8 Ks) existen.')

## Phase 1 — Entrenar HMM K∈{3..10} × 3 seeds

- **Input**: caches legacy en `cache/hmm_{ds}_K{k}.pth` (seed=42 implícita).
- **Output**: `cache/hmm_{ds}_K{k}_seed{s}.pth` para s ∈ {42, 2021, 7}.
- **Trabajo**: 64 Baum-Welch nuevos (seed=42 se copia desde legacy sin re-entrenar).
- **Resumible**: skip si el archivo existe.

Tiempo estimado CPU: **6-10 horas**. Puedes ejecutar en background:
```bash
nohup python -u scripts/paper1/phase1_train_hmm_seeds.py > logs/paper1_phase1.log 2>&1 &
tail -f logs/paper1_phase1.log
```

In [ ]:
# Dry-run: ver qué haría sin entrenar.
!python -u scripts/paper1/phase1_train_hmm_seeds.py --dry-run 2>&1 | tail -40

In [ ]:
# Ejecución real: entrena todo lo faltante. Live output en el notebook.
!python -u scripts/paper1/phase1_train_hmm_seeds.py

In [ ]:
# Verificar que las 96 caches (4 ds × 8 K × 3 seeds) están disponibles.
from scripts.paper1.config import hmm_cache_path, K_VALUES, SEEDS, DATASETS

missing = []
for ds in DATASETS:
    for K in K_VALUES:
        for s in SEEDS:
            p = Path(hmm_cache_path(ds['name'], K, s))
            if not p.exists():
                missing.append((ds['name'], K, s))

total = len(DATASETS) * len(K_VALUES) * len(SEEDS)
print(f'Caches disponibles: {total - len(missing)}/{total}')
if missing:
    print(f'\nFaltan {len(missing)}:')
    for m in missing[:10]:
        print(f'  {m}')

## Phase 2 — K-sweep downstream con 3 seeds

- **Input**: caches HMM de fase 1.
- **Output**: `scripts/paper1/k_optimal.json` con (variant, K) óptimo robusto por dataset.
- **Trabajo**: 4 datasets × 8 Ks × 2 variantes × 3 seeds = **192 experimentos**.
- **Resumible**: skip si `results/plan_a_..._ksweep_paper1_..._0/metrics.npy` existe.

Tiempo estimado CPU: **15-25 horas**. Background:
```bash
nohup python -u scripts/paper1/phase2_k_sweep_seeds.py > logs/paper1_phase2.log 2>&1 &
tail -f logs/paper1_phase2.log
```

In [ ]:
# Opcional: correr solo un subset para test rápido (ejemplo: ETTh1 K=4 seed=42).
# !python -u scripts/paper1/phase2_k_sweep_seeds.py --only-dataset ETTh1 --only-ks 4 --only-seed 42

In [ ]:
# Ejecución completa.
!python -u scripts/paper1/phase2_k_sweep_seeds.py

In [ ]:
# Inspeccionar k_optimal.json.
import json
k_opt = json.loads(Path('scripts/paper1/k_optimal.json').read_text())
print('K óptimo robusto por dataset (avg MSE sobre 3 seeds):')
for name, best in k_opt.items():
    if best.get('variant'):
        mses = ', '.join(f"{x:.4f}" for x in best['mse_per_seed'])
        print(f"  {name:12} {best['variant']:20} K={best['K']}  avg={best['mse_avg']:.6f} "
              f"seeds=[{mses}] n={best['n_seeds']}")
    else:
        print(f'  {name:12} SIN DATOS')

## Phase 3 — Plan A completo con 3 seeds

- **Input**: `k_optimal.json` de fase 2.
- **Output**: `results/plan_a_..._final_paper1_{technique}_..._seed{s}_0/metrics.npy`.
- **Trabajo**: 4 datasets × 6 técnicas × 4 horizontes × 3 seeds = **288 experimentos**.
- **Resumible**.

Tiempo estimado CPU: **60-80 horas**. Background recomendado:
```bash
nohup python -u scripts/paper1/phase3_plan_a_seeds.py > logs/paper1_phase3.log 2>&1 &
```

In [ ]:
# Opcional: correr solo un subset (ejemplo ETTh1 horizonte 96 seed 42).
# !python -u scripts/paper1/phase3_plan_a_seeds.py --only-dataset ETTh1 --only-horizons 96 --only-seed 42

In [ ]:
# Ejecución completa.
!python -u scripts/paper1/phase3_plan_a_seeds.py

## Phase 4 — Cross-domain Traffic/Exchange con 3 seeds

- **Input**: `k_optimal.json` + caches HMM.
- **Output**: `results/plan_a_..._crossdom_paper1_..._0/metrics.npy`.
- **Trabajo**: 2 targets × (5 baselines + 4 fuentes HMM) × 2 horizontes × 3 seeds = **108 experimentos**.
- **Resumible**.

Tiempo estimado CPU: **25-35 horas**.

In [ ]:
# Ejecución completa.
!python -u scripts/paper1/phase4_cross_domain_seeds.py

## Phase 5 — Métricas intrínsecas de tokenización con 3 seeds

- **Input**: caches HMM + `k_optimal.json`.
- **Output**: `results/paper1_intrinsic/{dataset}_seed{s}.json` con las 11 métricas por técnica.
- **Trabajo**: 4 datasets × 3 seeds = 12 configuraciones × 6 técnicas.
- **Resumible**.

Tiempo estimado CPU: **~30 min total** (mucho más barato que las fases downstream).

In [ ]:
# Ejecución completa.
!python -u scripts/paper1/phase5_intrinsic_metrics.py

In [ ]:
# Quick look a los resultados intrínsecos (ejemplo ETTh1 seed 42).
import json
fp = Path('results/paper1_intrinsic/ETTh1_seed42.json')
if fp.exists():
    data = json.loads(fp.read_text())
    for tech, metrics in data.items():
        if tech.startswith('_'):
            continue
        keys = ['compression_ratio', 'mse_reconstruction', 'acf_retention']
        vals = ', '.join(f'{k}={metrics.get(k, "-"):.4f}' if isinstance(metrics.get(k), (int, float)) else f'{k}={metrics.get(k, "-")}' for k in keys)
        print(f'{tech:20} {vals}')
else:
    print(f'{fp} no existe todavía. Ejecuta phase5.')

## Resumen de estado

Celda final para ver el progreso global del Paper-1 en cualquier momento.

In [ ]:
# Dashboard compacto: cuántos outputs por fase existen vs esperados.
from scripts.paper1.config import (
    DATASETS, CROSS_DOMAIN_TARGETS, K_VALUES, SEEDS, HORIZONS,
    HMM_VARIANTS, BASELINE_TECHNIQUES, CROSS_DOMAIN_HORIZONS, hmm_cache_path,
)

def count_files(pattern):
    return len(list(Path('.').glob(pattern)))

# Phase 1: caches HMM
expected_p1 = len(DATASETS) * len(K_VALUES) * len(SEEDS)
got_p1 = sum(
    Path(hmm_cache_path(ds['name'], K, s)).exists()
    for ds in DATASETS for K in K_VALUES for s in SEEDS
)
print(f'Phase 1 (HMM caches):      {got_p1}/{expected_p1}')

# Phase 2: K-sweep downstream
expected_p2 = len(DATASETS) * len(K_VALUES) * len(HMM_VARIANTS) * len(SEEDS)
got_p2 = count_files('results/plan_a_*_ksweep_paper1_*_0/metrics.npy')
print(f'Phase 2 (K-sweep runs):    {got_p2}/{expected_p2}')

# Phase 3: Plan A
expected_p3 = len(DATASETS) * (len(BASELINE_TECHNIQUES) + 1) * len(HORIZONS) * len(SEEDS)
got_p3 = count_files('results/plan_a_*_final_paper1_*_0/metrics.npy')
print(f'Phase 3 (Plan A runs):     {got_p3}/{expected_p3}')

# Phase 4: Cross-domain
expected_p4 = (
    len(CROSS_DOMAIN_TARGETS) * len(BASELINE_TECHNIQUES) * len(CROSS_DOMAIN_HORIZONS) * len(SEEDS)
    + len(CROSS_DOMAIN_TARGETS) * len(DATASETS) * len(CROSS_DOMAIN_HORIZONS) * len(SEEDS)
)
got_p4 = count_files('results/plan_a_*_crossdom_paper1_*_0/metrics.npy')
print(f'Phase 4 (Cross-domain):    {got_p4}/{expected_p4}')

# Phase 5: Intrinsic metrics
expected_p5 = len(DATASETS) * len(SEEDS)
got_p5 = count_files('results/paper1_intrinsic/*_seed*.json')
print(f'Phase 5 (Intrinsic):       {got_p5}/{expected_p5}')

# k_optimal.json
k_opt_exists = Path('scripts/paper1/k_optimal.json').exists()
print(f'\nk_optimal.json:            {"OK" if k_opt_exists else "FALTA (corre phase 2)"}')